# 🛡️ **Aegis-SRE: Phase 1 — Predictive Anomaly Detection**
### *Training the "Sentinel" using Classical Machine Learning*

---

> *This notebook serves as the training ground for the **Aegis-SRE Sentinel**, the predictive engine responsible for identifying system irregularities before they escalate into outages.*

---

## 🎯 The Objective

The goal of this phase is to ingest **over 1 million rows** of raw web server logs, perform high-density feature engineering, and train an unsupervised **Isolation Forest** model. This model will learn the *"normal" heartbeat* of a high-traffic e-commerce server and isolate anomalies such as:

| Anomaly Type | Detection Signal |
|---|---|
| 🔴 **Traffic Spikes & DDoS Patterns** | IP Frequency analysis |
| 🟡 **System Misconfigurations** | Status Code distribution |
| 🟠 **Data Leaks / Resource Exhaustion** | Payload Byte variance |

---

## 🧠 Why Isolation Forest?

In a production environment, system failures are **"few and different."** The Isolation Forest algorithm is uniquely suited for SRE tasks because:

- ✅ It **does not require pre-labeled** "error" data
- ✅ It **mathematically isolates outliers**, rather than profiling normality
- ✅ It is a robust, **unsupervised** choice for **zero-day anomaly detection**

---

## 📦 Output
- 🤖 aegis_anomaly_model.joblib
> *The serialized brain for our local monitoring agent.*

In [3]:
import kagglehub
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

###Load dataset

Web sever logs contain information on any event that was registered/logged. This contains a lot of insights on website visitors, behavior, crawlers accessing the site, business insights, security issues, and more.

This is a dataset for trying to gain insights from such a file.

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("eliasdabbas/web-server-access-logs")

print("Path to dataset files:", path)

100%|██████████| 267M/267M [00:02<00:00, 128MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/eliasdabbas/web-server-access-logs/versions/2


In [5]:
import os

os.listdir(path)

['access.log', 'client_hostname.csv']

In [7]:
log_file = os.path.join(path, os.listdir(path)[0])
print("Path to logfile: ", log_file)

Path to logfile:  /root/.cache/kagglehub/datasets/eliasdabbas/web-server-access-logs/versions/2/access.log


In [9]:
log_pattern = re.compile(
    r'(?P<ip>\S+) \S+ \S+ \[(?P<time>[^\]]+)\] "(?P<method>[A-Z]+) (?P<url>[^ "]+)? HTTP/[0-9.]+" (?P<status>\d{3}) (?P<bytes>\d+|-)'
)

In [12]:
parsed_data = []
limit = 1000000

print("Parsing logs....")

with open(log_file, "r", encoding="utf-8", errors="ignore") as file:
  for i, line in enumerate(file):
    if i >= limit:
      break

    match = log_pattern.search(line)
    if match:
      parsed_data.append(match.groupdict())

print("Completed: ", len(parsed_data))

Parsing logs....
Completed:  999985


In [13]:
df = pd.DataFrame(parsed_data)
display(df.head(10))

,ip,time,method,url,status,bytes
0,54.36.149.41,22/Jan/2019:03:56:14 +0330,GET,/filter/27|13%20%D9%85%DA%AF%D8%A7%D9%BE%DB%8C...,200,30577
1,31.56.96.51,22/Jan/2019:03:56:16 +0330,GET,/image/60844/productModel/200x200,200,5667
2,31.56.96.51,22/Jan/2019:03:56:16 +0330,GET,/image/61474/productModel/200x200,200,5379
3,40.77.167.129,22/Jan/2019:03:56:17 +0330,GET,/image/14925/productModel/100x100,200,1696
4,91.99.72.15,22/Jan/2019:03:56:17 +0330,GET,/product/31893/62100/%D8%B3%D8%B4%D9%88%D8%A7%...,200,41483
5,40.77.167.129,22/Jan/2019:03:56:17 +0330,GET,/image/23488/productModel/150x150,200,2654
6,40.77.167.129,22/Jan/2019:03:56:18 +0330,GET,/image/45437/productModel/150x150,200,3688
7,40.77.167.129,22/Jan/2019:03:56:18 +0330,GET,/image/576/article/100x100,200,14776
8,66.249.66.194,22/Jan/2019:03:56:18 +0330,GET,"/filter/b41,b665,c150%7C%D8%A8%D8%AE%D8%A7%D8%...",200,34277
9,40.77.167.129,22/Jan/2019:03:56:18 +0330,GET,/image/57710/productModel/100x100,200,1695


###Feature Engineering

In [16]:
df["bytes"] = pd.to_numeric(df["bytes"].replace("-", 0))
df["status"] = pd.to_numeric(df["status"])

In [15]:
df["time"] = pd.to_datetime(df["time"], format='%d/%b/%Y:%H:%M:%S %z', exact=False)
df["hour"] = df["time"].dt.hour
df["minute"] = df["time"].dt.minute

In [20]:
ip_counts = df["ip"].value_counts().to_dict()
df["ip_freq"] = df["ip"].map(ip_counts)

In [18]:
df["is_error"] = df["status"].apply(lambda x: 1 if x>=400 else 0)

In [27]:
features = ["bytes", "status", "hour", "ip_freq", "is_error"]
X = df[features]

print("Training ready features: \n")
display(X.head(10))

Training ready features: 



,bytes,status,hour,ip_freq,is_error
0,30577,200,3,23,0
1,5667,200,3,78,0
2,5379,200,3,78,0
3,1696,200,3,19,0
4,41483,200,3,5213,0
5,2654,200,3,19,0
6,3688,200,3,19,0
7,14776,200,3,19,0
8,34277,200,3,30860,0
9,1695,200,3,19,0


###Model Training

In [28]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import joblib

In [29]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("After Scaling: \n")
display(pd.DataFrame(X_scaled, columns=X.columns))

After Scaling: 



,bytes,status,hour,ip_freq,is_error
0,0.617932,-0.22618,-3.364766,-0.349955,-0.117806
1,-0.225285,-0.22618,-3.364766,-0.341134,-0.117806
2,-0.235034,-0.22618,-3.364766,-0.341134,-0.117806
3,-0.359706,-0.22618,-3.364766,-0.350597,-0.117806
4,0.987106,-0.22618,-3.364766,0.482434,-0.117806
...,...,...,...,...,...
999980,-0.333235,-0.22618,1.658976,-0.336964,-0.117806
999981,-0.339937,-0.22618,1.658976,-0.336964,-0.117806
999982,-0.336755,-0.22618,1.658976,-0.336964,-0.117806
999983,-0.332558,-0.22618,1.658976,-0.336964,-0.117806
